In [0]:
from pyspark.sql import functions as F

In [0]:
expenses = spark.read.csv(
    "/Volumes/pysparkazure_7405617424734388/default/week4/expenses.csv",
    header=True,
    inferSchema=True
)

display(expenses)

date,category,amount,notes
2026-01-03,Groceries,$1450.00,Weekly grocery run
2026-01-05,Rent,$12000.00,January rent
2026-01-08,Utilities,$850.50,Electricity bill
2026-01-10,Transport,$300.00,Auto fare
2026-01-15,Groceries,$980.00,Vegetables and fruits
2026-01-04,Rent,$11000.00,January rent
2026-01-06,Groceries,$1200.00,Monthly groceries
2026-01-12,Entertainment,$600.00,Movie night
2026-01-14,Utilities,$720.00,Water and electricity
2026-01-20,Healthcare,$450.00,Pharmacy


In [0]:
transactions_df = spark.read.csv(
    "/Volumes/pysparkazure_7405617424734388/default/week4/transactions.csv", header=True, inferSchema=True
)
display(transactions_df)

user_id,transaction_date,month,category,amount
1,2026-01-03,2026-01-01,Groceries,1450.0
1,2026-01-05,2026-01-01,Rent,12000.0
1,2026-01-08,2026-01-01,Utilities,850.5
1,2026-01-10,2026-01-01,Transport,300.0
1,2026-01-15,2026-01-01,Groceries,980.0
1,2026-01-25,2026-01-01,Shopping,9800.0
2,2026-01-04,2026-01-01,Rent,11000.0
2,2026-01-06,2026-01-01,Groceries,1200.0
2,2026-01-12,2026-01-01,Entertainment,600.0
2,2026-01-14,2026-01-01,Utilities,720.0


In [0]:
user_summary = transactions_df.groupBy("user_id", "month").agg(
    F.sum("amount").alias("monthly_spend")
)

In [0]:
user_details = spark.createDataFrame([
    (1, "Ravi Teja",   "ravi.teja@email.com",   "2025-01-15"),
    (2, "Sneha Reddy", "sneha.reddy@email.com",  "2025-02-10"),
    (3, "Arjun Kumar", "arjun.kumar@email.com",  "2025-01-22"),
    (4, "Priya Nair",  "priya.nair@email.com",   "2025-03-05"),
    (5, "Rohan Verma", "rohan.verma@email.com",  "2025-02-28"),
], ["user_id", "user_name", "email", "signup_date"])

In [0]:
monthly_budgets = spark.createDataFrame([
    (1, 12000.00),
    (2, 10000.00),
    (3, 11000.00),
    (4,  9000.00),
    (5, 13000.00),
], ["user_id", "monthly_budget"])

In [0]:
combined = transactions_df.join(
    user_details,
    on="user_id",
    how="inner"
)

combined.show(truncate=False)

+-------+----------------+----------+-------------+-------+-----------+---------------------+-----------+
|user_id|transaction_date|month     |category     |amount |user_name  |email                |signup_date|
+-------+----------------+----------+-------------+-------+-----------+---------------------+-----------+
|1      |2026-01-03      |2026-01-01|Groceries    |1450.0 |Ravi Teja  |ravi.teja@email.com  |2025-01-15 |
|1      |2026-01-05      |2026-01-01|Rent         |12000.0|Ravi Teja  |ravi.teja@email.com  |2025-01-15 |
|1      |2026-01-08      |2026-01-01|Utilities    |850.5  |Ravi Teja  |ravi.teja@email.com  |2025-01-15 |
|1      |2026-01-10      |2026-01-01|Transport    |300.0  |Ravi Teja  |ravi.teja@email.com  |2025-01-15 |
|1      |2026-01-15      |2026-01-01|Groceries    |980.0  |Ravi Teja  |ravi.teja@email.com  |2025-01-15 |
|1      |2026-01-25      |2026-01-01|Shopping     |9800.0 |Ravi Teja  |ravi.teja@email.com  |2025-01-15 |
|2      |2026-01-04      |2026-01-01|Rent     

In [0]:
monthly_summary = combined.groupBy("month", "user_id", "user_name", "email") \
    .agg(F.round(F.sum("amount"), 2).alias("total_spend"))

monthly_report = monthly_summary.join(monthly_budgets, on="user_id") \
    .withColumn("savings", F.round(F.col("monthly_budget") - F.col("total_spend"), 2)) \
    .withColumn(
        "alert_status",
        F.when(F.col("total_spend") > F.col("monthly_budget"), "Over Budget")
         .when(F.col("total_spend") < F.col("monthly_budget") * 0.7, "High Savings")
         .otherwise("On Track")
    ) \
    .select("month", "user_id", "user_name", "email",
            "total_spend", "monthly_budget", "savings", "alert_status") \
    .orderBy("month", "user_name")

monthly_report.show(truncate=False)

+----------+-------+-----------+---------------------+-----------+--------------+--------+------------+
|month     |user_id|user_name  |email                |total_spend|monthly_budget|savings |alert_status|
+----------+-------+-----------+---------------------+-----------+--------------+--------+------------+
|2026-01-01|3      |Arjun Kumar|arjun.kumar@email.com|13600.0    |11000.0       |-2600.0 |Over Budget |
|2026-01-01|1      |Ravi Teja  |ravi.teja@email.com  |25380.5    |12000.0       |-13380.5|Over Budget |
|2026-01-01|2      |Sneha Reddy|sneha.reddy@email.com|13970.0    |10000.0       |-3970.0 |Over Budget |
+----------+-------+-----------+---------------------+-----------+--------------+--------+------------+



In [0]:
category_summary = combined.groupBy("month", "user_name", "category") \
    .agg(F.round(F.sum("amount"), 2).alias("category_spend")) \
    .orderBy("month", "user_name", F.col("category_spend").desc())

category_summary.show(truncate=False)

+----------+-----------+-------------+--------------+
|month     |user_name  |category     |category_spend|
+----------+-----------+-------------+--------------+
|2026-01-01|Arjun Kumar|Rent         |9500.0        |
|2026-01-01|Arjun Kumar|Transport    |1800.0        |
|2026-01-01|Arjun Kumar|Entertainment|1200.0        |
|2026-01-01|Arjun Kumar|Groceries    |1100.0        |
|2026-01-01|Ravi Teja  |Rent         |12000.0       |
|2026-01-01|Ravi Teja  |Shopping     |9800.0        |
|2026-01-01|Ravi Teja  |Groceries    |2430.0        |
|2026-01-01|Ravi Teja  |Utilities    |850.5         |
|2026-01-01|Ravi Teja  |Transport    |300.0         |
|2026-01-01|Sneha Reddy|Rent         |11000.0       |
|2026-01-01|Sneha Reddy|Groceries    |1200.0        |
|2026-01-01|Sneha Reddy|Utilities    |720.0         |
|2026-01-01|Sneha Reddy|Entertainment|600.0         |
|2026-01-01|Sneha Reddy|Healthcare   |450.0         |
+----------+-----------+-------------+--------------+



In [0]:
alerts_only = monthly_report \
    .filter(F.col("alert_status") == "Over Budget") \
    .orderBy(F.col("total_spend").desc())

alerts_only.show(truncate=False)

+----------+-------+-----------+---------------------+-----------+--------------+--------+------------+
|month     |user_id|user_name  |email                |total_spend|monthly_budget|savings |alert_status|
+----------+-------+-----------+---------------------+-----------+--------------+--------+------------+
|2026-01-01|1      |Ravi Teja  |ravi.teja@email.com  |25380.5    |12000.0       |-13380.5|Over Budget |
|2026-01-01|2      |Sneha Reddy|sneha.reddy@email.com|13970.0    |10000.0       |-3970.0 |Over Budget |
|2026-01-01|3      |Arjun Kumar|arjun.kumar@email.com|13600.0    |11000.0       |-2600.0 |Over Budget |
+----------+-------+-----------+---------------------+-----------+--------------+--------+------------+



In [0]:
monthly_report.toPandas().to_csv(
    "monthly_report.csv",
    index=False
)

category_summary.toPandas().to_csv(
    "category_summary.csv",
    index=False
)

alerts_only.toPandas().to_csv(
    "budget_alerts.csv",
    index=False
)

print("Exported: monthly_report.csv")
print("Exported: category_summary.csv")
print("Exported: budget_alerts.csv")

Exported: monthly_report.csv
Exported: category_summary.csv
Exported: budget_alerts.csv


In [0]:
monthly_report.createOrReplaceTempView("monthly_report_view")

In [0]:
spark.sql("""
    SELECT user_name,
           COUNT(*)                         AS months_over_budget,
           ROUND(SUM(savings), 2)           AS total_overspend
    FROM   monthly_report_view
    WHERE  alert_status = 'Over Budget'
    GROUP  BY user_name
    ORDER  BY total_overspend ASC
""").show(truncate=False)

+-----------+------------------+---------------+
|user_name  |months_over_budget|total_overspend|
+-----------+------------------+---------------+
|Ravi Teja  |1                 |-13380.5       |
|Sneha Reddy|1                 |-3970.0        |
|Arjun Kumar|1                 |-2600.0        |
+-----------+------------------+---------------+



In [0]:
spark.sql("""
    SELECT month,
           ROUND(AVG(total_spend), 2) AS avg_spend_across_users
    FROM   monthly_report_view
    GROUP  BY month
    ORDER  BY month
""").show(truncate=False)

+----------+----------------------+
|month     |avg_spend_across_users|
+----------+----------------------+
|2026-01-01|17650.17              |
+----------+----------------------+

